In [7]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

df = pd.read_csv("HousePrices.csv")

def fit_model(data, target="Price"):
    X = data[["SqFt", "Bed", "Bath", "Offers"]]
    X = sm.add_constant(X)
    y = data[target]
    
    model = sm.OLS(y, X).fit()
    return model

In [8]:
df_listwise = df.dropna()

model_listwise = fit_model(df_listwise)

print(model_listwise.summary())

                            OLS Regression Results                            
Dep. Variable:                  Price   R-squared:                       0.683
Model:                            OLS   Adj. R-squared:                  0.671
Method:                 Least Squares   F-statistic:                     57.09
Date:                Sun, 29 Mar 2026   Prob (F-statistic):           1.34e-25
Time:                        12:28:39   Log-Likelihood:                -1220.2
No. Observations:                 111   AIC:                             2450.
Df Residuals:                     106   BIC:                             2464.
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const      -9814.6271   1.31e+04     -0.749      0.4

In [9]:
df_mean = df.copy()
mean_offers = df_mean["Offers"].mean()
df_mean["Offers"].fillna(mean_offers, inplace=True)

model_mean = fit_model(df_mean)

print(model_mean.summary())

                            OLS Regression Results                            
Dep. Variable:                  Price   R-squared:                       0.636
Model:                            OLS   Adj. R-squared:                  0.625
Method:                 Least Squares   F-statistic:                     53.84
Date:                Sun, 29 Mar 2026   Prob (F-statistic):           3.76e-26
Time:                        12:28:39   Log-Likelihood:                -1421.8
No. Observations:                 128   AIC:                             2854.
Df Residuals:                     123   BIC:                             2868.
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const      -1.175e+04   1.39e+04     -0.844      0.4

/var/folders/84/nq6m_5w937ddt23mzpd0k9h40000gn/T/ipykernel_82375/2497948065.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_mean["Offers"].fillna(mean_offers, inplace=True)


In [10]:
df_reg = df.copy()

known = df_reg[df_reg["Offers"].notna()]
missing = df_reg[df_reg["Offers"].isna()]

X_known = sm.add_constant(known[["SqFt", "Bed", "Bath"]])
y_known = known["Offers"]

reg_model = sm.OLS(y_known, X_known).fit()

X_missing = sm.add_constant(missing[["SqFt", "Bed", "Bath"]])
predicted_offers = reg_model.predict(X_missing)

df_reg.loc[df_reg["Offers"].isna(), "Offers"] = predicted_offers

model_reg = fit_model(df_reg)

print(model_reg.summary())

                            OLS Regression Results                            
Dep. Variable:                  Price   R-squared:                       0.628
Model:                            OLS   Adj. R-squared:                  0.616
Method:                 Least Squares   F-statistic:                     51.97
Date:                Sun, 29 Mar 2026   Prob (F-statistic):           1.47e-25
Time:                        12:28:39   Log-Likelihood:                -1423.2
No. Observations:                 128   AIC:                             2856.
Df Residuals:                     123   BIC:                             2871.
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const      -1.563e+04   1.41e+04     -1.107      0.2

In [11]:
df_stochastic = df.copy()

sigma = np.sqrt(reg_model.mse_resid)

stochastic_values = predicted_offers + np.random.normal(0, sigma, size=len(predicted_offers))

df_stochastic.loc[df_stochastic["Offers"].isna(), "Offers"] = stochastic_values

model_stochastic = fit_model(df_stochastic)

print(model_stochastic.summary())

                            OLS Regression Results                            
Dep. Variable:                  Price   R-squared:                       0.600
Model:                            OLS   Adj. R-squared:                  0.587
Method:                 Least Squares   F-statistic:                     46.17
Date:                Sun, 29 Mar 2026   Prob (F-statistic):           1.23e-23
Time:                        12:28:39   Log-Likelihood:                -1427.9
No. Observations:                 128   AIC:                             2866.
Df Residuals:                     123   BIC:                             2880.
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const      -1.119e+04   1.46e+04     -0.766      0.4

In [12]:
coefficients = []

for i in range(5):
    df_multi = df.copy()
    
    stochastic_values = predicted_offers + np.random.normal(0, sigma, size=len(predicted_offers))
    
    df_multi.loc[df_multi["Offers"].isna(), "Offers"] = stochastic_values
    
    model_multi = fit_model(df_multi)
    
    coefficients.append(model_multi.params)

coef_df = pd.DataFrame(coefficients)

avg_coef = coef_df.mean()

print(avg_coef)

const    -14042.351841
SqFt         53.808788
Bed        8926.584300
Bath      15621.269677
Offers   -11630.975131
dtype: float64
